# report

> Functions to aide reporting - need to document and show example of each.

In [ ]:
#| default_exp report

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
import pandas as pd
from sklearn.metrics import confusion_matrix
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import textwrap
from sklearn.pipeline import Pipeline
from supertree import SuperTree
import textwrap
from sklearn.utils.validation import check_is_fitted
from sklearn.exceptions import NotFittedError

In [ ]:
#| export
def preview_splits(X_train, y_train, X_test, y_test, label_names):
    print(f"Train: {len(X_train)} samples, {len(set(y_train))} classes")
    train_label_counts = pd.DataFrame(y_train).value_counts().to_frame()
    train_label_counts.insert(0, 'label_name', label_names)
    display(train_label_counts)
    print(f"Test: {len(X_test)} samples, {len(set(y_test))} classes")
    test_label_counts = pd.DataFrame(y_test).value_counts().to_frame()
    test_label_counts.insert(0, 'label_name', label_names)
    display(test_label_counts)

In [ ]:
#| export
def preview_label_counts(df, label_column, label_names):
    summary = pd.DataFrame(df.groupby([label_column])[label_column].count())
    summary.columns = ['count']
    summary.insert(0, 'label_name', label_names)
    display(summary)

In [ ]:
#| export
def preview_text_field(text: str, # Text to preview
					   width: int = 80 # Width to wrap the text to
					   ):
	""" Preview a text field, wrapping the text to 80 characters """
	for line in text.split("\r\n"):
		print(textwrap.fill(line, width=width))

In [ ]:
#| export
def preview_row_text(df: pd.DataFrame, # DataFrame containing the data
					 selected_index: int, # Index of the row to preview 
					 text_column: str = 'text', # column name for text field
					 limit: int = -1 # Limit the length of the text field
					 ):
	""" Preview the text fields of a row in the DataFrame """

	if selected_index not in df.index:
		print(f"Index {selected_index} not in DataFrame")
		return

	summary = df.loc[selected_index].to_frame().drop(text_column)
	summary.columns = ['Value']
	summary.index.name = 'Attribute'
	display(summary)

	print(f"{text_column}:")
	text = df[text_column].loc[selected_index]
	if limit > 1:
		if len(text) > limit:
			text = text[:limit] + "..."
	preview_text_field(text)

In [ ]:
#| export
def plot_confusion_matrix(y_test, y_predicted, target_classes, target_names):
	# Compute confusion matrix
	cm = confusion_matrix(y_test, y_predicted, labels=target_classes)

	# Compute row and column totals
	row_totals = cm.sum(axis=1)  # Row totals
	col_totals = cm.sum(axis=0)  # Column totals
	overall_total = cm.sum()  # Overall total

	# Compute normalized proportions
	cm_normalized = cm / cm.sum(axis=1, keepdims=True)  # Normalize rows (proportions)

	# Combine counts and proportions into annotations
	annotations = np.empty_like(cm).astype(str)
	for i in range(cm.shape[0]):
		for j in range(cm.shape[1]):
			annotations[i, j] = f"{cm[i, j]}\n({cm_normalized[i, j]:.2f})"

	# Create updated axis labels with totals
	xticklabels_with_totals = [f"{label}\n(Total: {total})" for label, total in zip(target_names, col_totals)]
	yticklabels_with_totals = [f"{label} (Total: {total})" for label, total in zip(target_names, row_totals)]

	# Create heatmap without totals in the matrix
	fig, ax = plt.subplots(figsize=(10, 8))
	sns.heatmap(cm,
				annot=annotations,
				fmt='',
				cmap='Blues',
				xticklabels=xticklabels_with_totals,
				yticklabels=yticklabels_with_totals,
				cbar=True)

	plt.xlabel('Predicted Labels')
	plt.ylabel('Actual Labels')

	plt.show()


In [ ]:
#| export
def plot_decision_tree_from_pipeline(pipeline, # The pipeline containing the classifier
									X_train, # The training data
					   				y_train, # The training labels
					   				target_names, # The target names
									classifier_step_name = 'classifier', # The name of the classifier step in the pipeline
									features_step_name = 'features', # The name of the final preprocessing step = probably the name of the step prior to the classifier
					): # outputs a tree plot
	""" Plot the decision tree of the classifier from a pipeline using SuperTree """

	preprocessor = Pipeline(pipeline.steps[:-1])
	X_train_preprocessed = preprocessor.fit_transform(X_train, y_train)
	feature_names = preprocessor.named_steps[features_step_name].get_feature_names_out()
	super_tree = SuperTree(pipeline.named_steps[classifier_step_name], X_train_preprocessed.toarray(), y_train, feature_names, target_names)
	super_tree.show_tree()

In [ ]:
#| export
def get_classifier_feature_names_in(pipeline, classifier_step_name = 'classifier'):
	feature_names = None
	for i, step in enumerate(pipeline.named_steps):
		if hasattr(pipeline.named_steps[step], 'get_feature_names_out'):
			feature_names = pipeline.named_steps[step].get_feature_names_out(feature_names)
		if step == classifier_step_name:
			return feature_names

In [ ]:
#| export
def plot_logistic_regression_features_from_pipeline(pipeline, target_classes, target_names, top_n=20, classifier_step_name='classifier', features_step_name='features'):
	# Get the classifier and feature names
	classifier = pipeline.named_steps[classifier_step_name]
	feature_names = get_classifier_feature_names_in(pipeline, classifier_step_name)

	# for binary classification
	if len(classifier.classes_) == 2:  # Binary classification
		log_odds = classifier.coef_[0]  # Single row for binary classification
		odds_ratio = np.exp(log_odds)  # Convert log odds to odds ratio

		feature_importance = pd.DataFrame({
			'Feature': feature_names,
			'Log Odds (Logit)': log_odds,
			'Odds Ratio': odds_ratio
		}).sort_values(by='Log Odds (Logit)', ascending=False)

		feature_importance['abs_log_odds'] = np.abs(feature_importance['Log Odds (Logit)'])
		feature_importance = feature_importance.sort_values(by='abs_log_odds', ascending=False).head(top_n)
		feature_importance = feature_importance.drop(columns=['abs_log_odds'])

		if len(feature_importance) < top_n:
			top_n = len(feature_importance)

		plt.figure(figsize=(10, 6))
		sns.barplot(x='Log Odds (Logit)', y='Feature', data=feature_importance.head(top_n))
		plt.title(f"Top {top_n} Features for Binary Classification")
		plt.xlabel("Log Odds (Logit)")
		plt.ylabel("Feature")
		plt.tight_layout()
		plt.show()

		# Display the feature importance DataFrame
		display(feature_importance.head(top_n))

	else:  # Multi-class classification
		for class_idx, class_name in enumerate(classifier.classes_):
			class_name = target_names[class_idx]
			log_odds = classifier.coef_[class_idx]  # Coefficients for the current class
			odds_ratio = np.exp(log_odds)  # Convert log odds to odds ratio

			# Create a DataFrame for feature importance
			feature_importance = pd.DataFrame({
				'Feature': feature_names,
				'Log Odds (Logit)': log_odds,
				'Odds Ratio': odds_ratio
			}).sort_values(by='Log Odds (Logit)', ascending=False)

			if len(feature_importance) < top_n:
				top_n = len(feature_importance)

			plt.figure(figsize=(10, 6))
			sns.barplot(x='Log Odds (Logit)', y='Feature', data=feature_importance.head(top_n))
			plt.title(f"Top {top_n} Features for Class: {class_name}")
			plt.xlabel("Log Odds (Logit)")
			plt.ylabel("Feature")
			plt.tight_layout()
			plt.show()

			# Display the feature importance DataFrame
			display(feature_importance.head(top_n))

In [ ]:
#| export
def get_selected_feature_names(pipeline, # the pipeline to get the feature names from
							   features_step_name = 'features', # the name of the step in the pipeline that contains the features 
							   selector_step_name = 'selector', # the name of the step in the pipeline that contains the selector
							   ) -> list: # returns a list of the selected feature names
	""" Get the selected features from the pipeline. """

	feature_names = pipeline.named_steps[features_step_name].get_feature_names_out()

	if hasattr(pipeline.named_steps[selector_step_name], 'get_support'):
		selected_features = pipeline.named_steps[selector_step_name].get_support()
	else:
		selected_features = pipeline.named_steps[selector_step_name].support_

	selected_feature_names = []
	for i, include in enumerate(selected_features):
		if include:
			selected_feature_names.append(feature_names[i])

	return selected_feature_names

In [ ]:
#| export
def preview_selected_features(pipeline, # the pipeline to preview the selected features from
							   features_step_name = 'features', # the name of the step in the pipeline that contains the features 
							   selector_step_name = 'selector', # the name of the step in the pipeline that contains the selector
							   ):
	""" Preview (i.e. prints) the selected features from the pipeline. """
	selected_feature_names = get_selected_feature_names(pipeline, features_step_name, selector_step_name)
	if len(selected_feature_names) == 0:
		print("No features selected.")
	else:
		for feature in selected_feature_names:
			print(feature)

In [ ]:
#| export
def preview_pipeline_features(pipeline):
	try:
		check_is_fitted(pipeline)
	except NotFittedError as e:
		raise NotFittedError('This pipeline is not fitted. Fit it before invoking preview_pipeline_features.')

	feature_names = None
	for i, step in enumerate(pipeline.named_steps):
		print(f'Step {i}: {pipeline.named_steps[step]}')
		if hasattr(pipeline.named_steps[step], 'get_feature_names_out'):
			feature_names = pipeline.named_steps[step].get_feature_names_out(feature_names)
			print(f'Features ({len(feature_names)}):')
			print(textwrap.fill(", ".join(feature_names), 100, initial_indent = '\t', subsequent_indent = '\t'))
		print()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()